28. 找出字符串中第一个匹配项的下标

https://leetcode.cn/problems/find-the-index-of-the-first-occurrence-in-a-string/description/?envType=study-plan-v2&envId=top-interview-150

给你两个字符串 haystack 和 needle ，请你在 haystack 字符串中找出 needle 字符串的第一个匹配项的下标（下标从 0 开始）。如果 needle 不是 haystack 的一部分，则返回 -1 。

以下是我写的代码，我的想法是以 needle 线性搜索为条件，不断过滤 haystack 中的起始点。这种做法最坏的情况是，每个位置都自始至终匹配，例如 `aaaaaa` 和 `aa` 就不会在一开始就返回 0；写着写着我自己都逗笑了，我的算法对于简单情况的处理非常糟糕，最坏情况下的复杂度是 o(m*n)。也就是说，实际上我想出的算法没比普通的匹配快多少.但是私以为，该算法对于重复字符比较少的（例如自然语言文本）的性能不会更差。实际上，本题的 KMP 算法是在大学课堂就讲过的内容，但是经年不用已经忘干净了。本题作为一复习。

In [1]:
class Solution:
    def strStr(self, haystack: str, needle: str) -> int:
        if len(haystack) < len(needle) or len(needle) == 0:
            return -1
        candidates = set(range(len(haystack)))
        for i in range(len(needle)):
            if not candidates:
                return -1
            new_candidates = set()
            for j in candidates:
                if j + i > len(haystack) - 1:
                    continue
                elif haystack[j + i] == needle[i]:
                    new_candidates.add(j)
            candidates = new_candidates
        if not candidates:
            return -1
        else:
            answer = candidates.pop()
            while candidates:
                answer = min(candidates.pop(), answer)
            return answer
        

在正式开始前，让我们先证明一个微不足道的事实。假设 needle 符合某种 prefix-suffix 结构，即 x-y-x-m；其中 x 是最长的重复子串。

现在，如果 x 向右平移一个单位 x[1:] 后，其还能与 x 本身按位相等，说明什么？假设 x 的最后一位是字母 a，那么这说明倒数第二位也是字母 a，而倒数第三位也是。这是唯一的可能性。

```
a a a a
\|\|\|\
  a a a a
```

同理，如果 x 向右平移两个单位后还能对上，假设那两位是 ab，那么字符串一定是 abababab。因此，如果一个字符串可以平移后按位相等，则其形如 x = ppp 结构，其中 p 可以是对应平移长度的子串。

现在，我们可以证明，当 x 是最长的重复子串时，x-y 不可能出现平移后相等，也就是说，x-y-x 中不能再规范并形式化为 z-x-y-m，其中 len(m) + len(z) = len(x)。因为如果出现，那么意味着平移的部分 z 是 x 的组成单元，意味着 x 本身还可以更长，即 x = z + x。而这是矛盾的。

因此，如果我们已经在 needle 中发现了 x-y-x-m 结构，并且 haystack 在 m 部分出现了问题；那么，在 haystack 部分的指针无需再从 x-y 这一部分的**任何位置**开始比较，因为其中都不可能再出现 x-y 这一子串结构。因此，只需要从 x suffix 开始。

与此同时，x suffix 也是比较过的部分；因此，在 needle 的指针只需要从 x prefix 结尾处开始比较，在 haystack 的指针则看似原地不动了。而这就是 KMP 的核心思想。

**将 haystack 上的回溯转变为在 needle 上检查前缀和后缀是否有重复**，这就是精髓。构建一个与 needle 等长的 lps（最长相等前后缀数组），这个数组存储的就是，当前位置的下一个位置如果出现不匹配了，那么退回到先前的哪个位置重新去匹配。

In [2]:
needle = "ababaabab"
m = len(needle)
lps = [0] * m
j = 0

for i in range(1, m):
    while j > 0 and needle[i] != needle[j]:
        j = lps[j - 1]
    if needle[i] == needle[j]:
        j += 1
        lps[i] = j

print(list(range(len(needle))))
print(lps)
print(list(needle))

[0, 1, 2, 3, 4, 5, 6, 7, 8]
[0, 0, 1, 2, 3, 1, 2, 3, 4]
['a', 'b', 'a', 'b', 'a', 'a', 'b', 'a', 'b']


In [5]:
class Solution:
    def strStr(self, haystack: str, needle: str) -> int:
        if needle == "":
            return 0

        m = len(needle)
        lps = [0] * m
        j = 0

        for i in range(1, m):
            while j > 0 and needle[i] != needle[j]:
                j = lps[j - 1]
            if needle[i] == needle[j]:
                j += 1
                lps[i] = j

        j = 0
        for i, ch in enumerate(haystack):
            while j > 0 and ch != needle[j]:
                j = lps[j - 1]
            if ch == needle[j]:
                j += 1
                if j == m:
                    return i - m + 1
        return -1
    
solution = Solution()
haystack = "abaab"
needle = "aa"
solution.strStr(haystack, needle)

2